# AAV Snapback Breakpoint and RNAfold Analysis

This notebook analyzes AAV snapback breakpoint patterns from long-read sequencing subparser output.

## Project Goals

- Parse snapback tile-count output from the subparser  
- Extract breakpoint coordinates from plus/coding and minus/non-coding strand payload tiles  
- Generate breakpoint sequence windows  
- Run RNAfold to predict local secondary structure  
- Extract minimum free energy (MFE) values  
- Merge MFE with breakpoint abundance  
- Add breakpoint nucleotide identity  
- Test nucleotide enrichment compared with payload background  
- Generate final tables and visualizations  

## 1. Import libraries

These packages are used for file handling, parsing, sequence analysis, statistics, plotting, and RNAfold execution.


In [ ]:
%load_ext rpy2.ipython
import re
import os
import pandas as pd
from Bio import SeqIO
import shutil
from Bio.SeqRecord import SeqRecord
import pandas as pd
import subprocess
from pathlib import Path
import os
import numpy as np
from scipy.stats import chisquare

## 2. Path configuration
Update these paths only before running on your own machine.

In [ ]:
from pathlib import Path

# =========================================================
# USER PATH CONFIGURATION (EDIT ONLY THIS SECTION)
# =========================================================

PROJECT_DIR = Path("/path/to/your/project")

INPUT_TILE_FILE = PROJECT_DIR / "data/input_tile_files/sample_tile_file.txt"

REFERENCE_FASTA = PROJECT_DIR / "data/reference_fasta/reference_payload.fa"

OUTPUT_FOLDER = PROJECT_DIR / "results/sample_output"

# Main pipeline input structure
file_reference_pairs = [
    (
        INPUT_TILE_FILE,
        REFERENCE_FASTA,
        OUTPUT_FOLDER
    ),
]

print("Input tile file:", INPUT_TILE_FILE)
print("Reference file:", REFERENCE_FASTA)
print("Output folder:", OUTPUT_FOLDER)

## 3. Core breakpoint processing workflow

This section performs the complete breakpoint analysis workflow including:

- parsing subparser output  
- extracting plus/minus breakpoint coordinates  
- generating payload FASTA files  
- creating reverse complement sequences  
- generating breakpoint sequence windows  
- running RNAfold  
- extracting MFE values  
- merging structural predictions with breakpoint abundance  

In [ ]:
for file_name, ref_filename, output_folder in file_reference_pairs:
    # Reset output folder if rerunning
    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)

    os.makedirs(output_folder, exist_ok=True)    

    file = open(file_name, "r")
    line = file.readline().strip().replace("e-", "e")
    # Use os.path.basename to get the filename if file_name is a full file path
    parts = os.path.basename(file_name).rsplit('.', 1) #basename select the filename and here rsplit('.',1) say split once and that is done based on last dot
    base_name = parts[0]  # This will always get the filename without its last extension
    output_file_path = os.path.join(output_folder, base_name + ".csv")
    # Print the output file path
    print(f"Writing to file: {output_file_path}")
    outfile = open(output_file_path, "w+")
    outfile.write("Count,Proportion,StartA,EndA,StrandA,StartB,EndB,StrandB\n")
    pattern = r"Payload\S*\[(\d+)-(\d+)\]\(([tf\+\-])\)"
    while line:
        matches = re.findall(pattern, line)
        if len(matches) >= 2:
            startA, endA, strandA = matches[0]
            startB, endB, strandB = matches[1]
            # Normalize strands to just 't' or 'f'
            strandA = 't' if strandA in ['t', '+'] else 'f'
            strandB = 't' if strandB in ['t', '+'] else 'f'
            count, proportion = line.split()[0], line.split()[1]
            to_append = f"{count},{proportion},{startA},{endA},{strandA},{startB},{endB},{strandB}\n"
            outfile.write(to_append)
        line = file.readline().strip().replace("e-", "e")
    outfile.close()
    file.close()
    # WRITE THE NAME OF FILE which has all point, same as name mentioned above
    filename = os.path.basename(file_name)  # Extracts the filename from the path
    path = os.path.splitext(filename)[0] + ".csv"  # Replaces the file extension with .csv
    full_path = os.path.join(output_folder, path)
    print(full_path)
    # Read and process plus strand data
    data_plus = pd.read_csv(full_path)
    data_plus = data_plus[["Count", "EndA", "StrandA"]]
    data_plus = data_plus[data_plus["StrandA"] == 't']
    data_plus = data_plus.groupby(['EndA'], as_index=False).sum(numeric_only=True)
    add_count = data_plus['Count'].sum()
    data_plus['Proportion'] = data_plus['Count'] / add_count
    outfile_path = os.path.join(output_folder, "PROPORTION_enda.csv")
    # Save the data_plus DataFrame as a CSV file
    data_plus.to_csv(outfile_path, index=False)
    # Read and process minus strand data
    data_minus = pd.read_csv(full_path)
    data_minus = data_minus[["Count", "StartB", "StrandB"]]
    data_minus = data_minus[data_minus["StrandB"] == 't']
    data_minus = data_minus.groupby(['StartB'], as_index=False).sum(numeric_only=True)
    add_count1 = data_minus['Count'].sum()
    data_minus['Proportion'] = data_minus['Count'] / add_count1
    outfile_path = os.path.join(output_folder, "PROPORTION_startb.csv")
    # Save the data_plus DataFrame as a CSV file
    data_minus.to_csv(outfile_path, index=False)

    # Extracting Payload part from the reference file and making reverse complement file
    # Function to generate a reverse complement sequence record
    def make_rc_record(record):
        return SeqRecord(seq=record.seq.reverse_complement(), \
                         id="{}-reversecomplement".format(record.id), \
                         description="Reverse complement")

    # Read the reference file and select the payload sequence
    sequences = [i for i in SeqIO.parse(ref_filename, 'fasta')]
    payload_sequence = None
    for sequence in sequences:
        if sequence.name == 'Payload':
            payload_sequence = sequence

    # Save the plus strand payload sequence to a file
    plusstrand_output_path = os.path.join(output_folder, "plusstrand.fasta")
    SeqIO.write(payload_sequence, plusstrand_output_path, 'fasta')

    # Generate reverse complement records for the plus strand payload sequence
    revcomp_records = map(make_rc_record, SeqIO.parse(plusstrand_output_path, "fasta"))

    # Save the reverse complement records to the output file in FASTA format
    revcomp_output_path = os.path.join(output_folder, "revcomp_minusstrand.fasta")
    SeqIO.write(revcomp_records, revcomp_output_path, 'fasta')

    # Modify the plusstrand.fasta and revcomp_minusstrand.fasta files
    payload_plusstrand_output_path = os.path.join(output_folder, "payload_plusstrand.fasta")
    payload_minusstrand_output_path = os.path.join(output_folder, "payload_minusstrand.fasta")

    with open(plusstrand_output_path, 'r') as f_in, open(payload_plusstrand_output_path, 'w') as f_out:
        for line in f_in:
            if line.startswith(">Payload"):
                f_out.write(">Payload\n")
            else:
                f_out.write(line)

    # Perform file operations to modify the plusstrand.fasta file
    input_filename = "payload_plusstrand.fasta"
    output_filename = "Comb_plus_brkp.seq"
    csv_filename = "PROPORTION_enda.csv"
    path = os.path.join(output_folder, csv_filename)
    data = pd.read_csv(path)
    mylist = data['EndA'].tolist()

    # Construct the output file path within the output folder
    output_file = os.path.join(output_folder, output_filename)

    with open(revcomp_output_path, 'r') as f_in, open(payload_minusstrand_output_path, 'w') as f_out:
        for line in f_in:
            if line.startswith(">Payload"):
                f_out.write(">Payload\n")
            else:
                f_out.write(line)
    
    # Perform file operations to modify the plusstrand.fasta file
    input_filename = "payload_plusstrand.fasta"
    output_filename = "Comb_plus_brkp.seq"
    csv_filename = "PROPORTION_enda.csv"
    path = os.path.join(output_folder, csv_filename)
    data = pd.read_csv(path)
    mylist = data['EndA'].tolist()

    # Construct the output file path within the output folder
    output_file = os.path.join(output_folder, output_filename)

    with open(output_file, 'w') as plus:
        for i in mylist:
            input_filepath = os.path.join(output_folder, input_filename)
            with open(input_filepath, 'r') as input_file:
                payload = input_file.read().replace('>Payload', ' ').replace('\n', '')

            Seq = [
                f"> {i}_10bp-Brkp-10bp\n{payload[i-10:i+11]}",
                f"> {i}_15bp-Brkp-15bp\n{payload[i-15:i+16]}",
                f"> {i}_20bp-Brkp-20bp\n{payload[i-20:i+21]}",
                f"> {i}_40bp-Brkp\n{payload[i:i+41]}",
                f"> {i}_Brkp-40bp\n{payload[i-40:i+1]}",
                f"> {i}_5bp-Brkp-35bp\n{payload[i-5:i+36]}",
                f"> {i}_35bp-Brkp-5bp\n{payload[i-35:i+6]}",
                f"> {i}_10bp-Brkp-30bp\n{payload[i-10:i+31]}",
                f"> {i}_15bp-Brkp-25bp\n{payload[i-15:i+26]}",
                f"> {i}_25bp-Brkp-15bp\n{payload[i-25:i+16]}",
                f"> {i}_30bp-Brkp-10bp\n{payload[i-30:i+11]}"
            ]
            result = '\n'.join(Seq)
            plus.write(result + "\n")

    # Perform file operations to remove empty lines with their headers
    input_filename = "Comb_plus_brkp.seq"
    output_filename = "Final_plus_Comb_brkp.seq"
    file_path = os.path.join(output_folder, input_filename)
    output_path = os.path.join(output_folder, output_filename)

    with open(file_path, 'r') as file:
        lines = file.readlines()

    cleaned_lines = []
    header_line = None

    for line in lines:
        line = line.strip()
        if line.startswith(">"):
            header_line = line
        elif line == "":
            header_line = None
        elif header_line is not None:
            cleaned_lines.append(header_line)
            cleaned_lines.append(line)
            header_line = None

    with open(output_path, 'w') as file:
        for line in cleaned_lines:
            file.write(line + '\n')

    # Extract sequnce of different windows for minus strand
    input_filename = "payload_minusstrand.fasta"
    output_filename = "Comb_minus_brkp.seq"
    csv_filename = "PROPORTION_startb.csv"
    path = os.path.join(output_folder, csv_filename)
    data = pd.read_csv(path)
    mylist = data['StartB'].tolist()
    
    # Construct the output file path within the output folder
    output_file = os.path.join(output_folder, output_filename)

    def split_and_expand(row, sep, n):
        parts = row.split(sep, n)
        # Ensure there are n+1 parts after the split
        parts += [''] * (n + 1 - len(parts))
        return parts


    with open(output_file, 'w') as minus:
        for i in mylist:
            input_filepath = os.path.join(output_folder, input_filename)
            with open(input_filepath, 'r') as input_file:
                payload = input_file.read().replace('>Payload', ' ').replace('\n', '')

            Seq = [
                f"> {i}_10bp-Brkp-10bp\n{payload[i-10:i+11]}",
                f"> {i}_15bp-Brkp-15bp\n{payload[i-15:i+16]}",
                f"> {i}_20bp-Brkp-20bp\n{payload[i-20:i+21]}",
                f"> {i}_40bp-Brkp\n{payload[i:i+41]}",
                f"> {i}_Brkp-40bp\n{payload[i-40:i+1]}",
                f"> {i}_5bp-Brkp-35bp\n{payload[i-5:i+36]}",
                f"> {i}_35bp-Brkp-5bp\n{payload[i-35:i+6]}",
                f"> {i}_10bp-Brkp-30bp\n{payload[i-10:i+31]}",
                f"> {i}_15bp-Brkp-25bp\n{payload[i-15:i+26]}",
                f"> {i}_25bp-Brkp-15bp\n{payload[i-25:i+16]}",
                f"> {i}_30bp-Brkp-10bp\n{payload[i-30:i+11]}"
            ]
            result = '\n'.join(Seq)
            minus.write(result + "\n")

    # Perform file operations to remove empty lines with their headers
    input_filename = "Comb_minus_brkp.seq"
    output_filename = "Final_minus_Comb_brkp.seq"
    file_path = os.path.join(output_folder, input_filename)
    output_path = os.path.join(output_folder, output_filename)

    with open(file_path, 'r') as file:
        lines = file.readlines()

    cleaned_lines = []
    header_line = None

    for line in lines:
        line = line.strip()
        if line.startswith(">"):
            header_line = line
        elif line == "":
            header_line = None
        elif header_line is not None:
            cleaned_lines.append(header_line)
            cleaned_lines.append(line)
            header_line = None

    with open(output_path, 'w') as file:
        for line in cleaned_lines:
            file.write(line + '\n')
    
    
    def make_rc_record(record):
        """Returns a new SeqRecord with the reverse complement sequence."""
        return SeqRecord(seq=record.seq.reverse_complement(),
                         id= " " + record.id,
                         description="")

    input_file = os.path.join(output_folder, "Final_minus_Comb_brkp.seq")
    output_file = "rc_minusstrand.fasta"

    # Read the input file and generate reverse complement records
    records = (make_rc_record(record) for record in SeqIO.parse(input_file, "fasta"))

    # Construct the output file path within the output folder
    output_path = os.path.join(output_folder, output_file)

    # Save the reverse complement records to the output file in FASTA format
    with open(output_path, "w") as output_handle:
        SeqIO.write(records, output_handle, "fasta")
   
    # Running RNAFOLD software for plus strand
    file_name1 = "Final_plus_Comb_brkp.seq"
    file_path1 = os.path.join(output_folder, file_name1)

    plus_folder = os.path.join(output_folder, 'plus_folder')
    os.makedirs(plus_folder, exist_ok=True)

    # Save current directory
    curr_dir = os.getcwd()

    # Change to output directory
    os.chdir(plus_folder)

    os.system(f"RNAfold -p -d2 --noLP < {file_path1} > Comb_plus_rnafold.seq")

    # Change back to original directory
    os.chdir(curr_dir)

    for file in os.listdir(plus_folder):
        if file.endswith(".ps"):
            shutil.move(os.path.join(plus_folder, file), os.path.join(plus_folder, file))


    # Running RNAFOLD software for minus strand
    file_name2 = "rc_minusstrand.fasta"
    file_path2 = os.path.join(output_folder, file_name2)

    minus_folder = os.path.join(output_folder, 'minus_folder')
    os.makedirs(minus_folder, exist_ok=True)

    # Change to output directory
    os.chdir(minus_folder)

    os.system(f"RNAfold -p -d2 --noLP < {file_path2} > Comb_minus_rnafold.seq")

    # Change back to original directory
    os.chdir(curr_dir)

    for file in os.listdir(minus_folder):
        if file.endswith(".ps"):
            shutil.move(os.path.join(minus_folder, file), os.path.join(minus_folder, file))
    # Move Comb_minus_rnafold.seq to the main output_folder
    shutil.move(os.path.join(minus_folder, "Comb_minus_rnafold.seq"), output_folder)

    # Move Comb_plus_rnafold.seq to the main output_folder
    shutil.move(os.path.join(plus_folder, "Comb_plus_rnafold.seq"), output_folder)
    
    # Lets pick only those header that will have similar MFE STRUCTURE and Centroid Structure
    #Reason (it gives us more confidence that they are true secondary structure ):
    #with open(r'file.fasta', 'w') as fp:
    plusrnafold_filepath = os.path.join(output_folder, "Comb_plus_rnafold.seq")
    with open(plusrnafold_filepath,'r') as fd:
        seq_list = fd.readlines() # convert above line of files as a list and read it.
        list1 = [] #creating empty list 1
        list2 = [] #creating empty list 2
        char = " " #character based on which we seperate the lines
        for i in range(2,len(seq_list),6): #creating loop and picking every 2,3,4 line from each batch
            seq1=seq_list[i] # picking line2 example: .((((.......))))......................... 
            seq1_indx=seq1.index(char) #finding position of first index character
            seq1_substr = seq1[:seq1_indx] #picking substring till indexed character
            seq2=seq_list[i+2] # picking line4 example: .((((.......)))).........................
            seq2_indx=seq2.index(char) #finding position of first index character
            seq2_substr = seq2[:seq2_indx] #picking till indexed character
            if seq1_substr==seq2_substr: #check if both lines are same or not
                list1.append(seq_list[i-2].strip()) #appending line2
                list2.append(seq_list[i].strip()) #appending line1
    df = pd.DataFrame(list(zip(list1, list2)),columns =['Brk', 'val']) #Calling DataFrame after zipping both lists with names
    #df = pd.DataFrame(list(zip(list1, list2)),columns =['Brk', 'val']) #Calling DataFrame after zipping both lists with names
    df[['sep','Sliding_window']] = df["Brk"].str.split(">",n=1, expand = True)
    #Split the 'Brk' column string by spaces ' ', then take the second part (index 1),and split this part by '_' and take the first part (index 0)
    df['EndA'] = df['Brk'].str.split(' ').str[1].str.split('_').str[0]
    # Split the 'val' column string by '(' first, then by ')', and finally by ' ', we use rsplit to split from the right and only split once (maxsplit parameter is set to 1)
    df['MFE'] = df['val'].str.rsplit('(', n=1).str[1].str.split(')').str[0].str.strip()
    df['MFE'] = df['MFE'].astype(str).str.replace('-', '')
    df['MFE'] = df['MFE'].astype(str).astype(float)
    df['ranked']=df.groupby('EndA')['MFE'].rank('dense',ascending=False) # grouping based on breakpoint and choosing the one with highest MFE as we replaced minus sign, this can be called as lowest MFE
    df_final=df[df['ranked']==1] #(We only pick those whose MFE is lowest)
    # Select the columns of interest
    Column_of_Interest = df_final.loc[:, ['EndA', 'MFE', 'ranked', 'Sliding_window']]
    Drp_duplicates_column_of_Intrst_plus = Column_of_Interest.drop_duplicates(subset='EndA', keep='first') #Only keep top values of MFE and drop other values
    Drp_duplicates_column_of_Intrst_plus = pd.DataFrame(Drp_duplicates_column_of_Intrst_plus)
    outfile_path = os.path.join(output_folder, "Drp_duplicates_column_of_Intrst_plus.csv")
    # Save the data_plus DataFrame as a CSV file
    Drp_duplicates_column_of_Intrst_plus.to_csv(outfile_path, index=False)
    # Lets pick only those header that will have similar MFE STRUCTURE and Centroid Structure
    #Reason (it gives us more confidence that they are true secondary structure ):
    #with open(r'file.fasta', 'w') as fp:
    # A more robust way to handle the split operation
    minusrnafold_filepath = os.path.join(output_folder, "Comb_minus_rnafold.seq")
    with open(minusrnafold_filepath,'r') as fd:
        seq_list = fd.readlines() # convert above line of files as a list and read it.
        #print(seq_list) #print the list
        #print(type(seq_list)) #print type of list
        list1 = [] #creating empty list 1
        list2 = [] #creating empty list 2
        char = " " #character based on which we seperate the lines
        for i in range(2,len(seq_list),6): #creating loop and picking every 2,3,4 line from each batch
            seq1=seq_list[i] # picking line2 example: .((((.......))))......................... 
            seq1_indx=seq1.index(char) #finding position of first index character
            seq1_substr = seq1[:seq1_indx] #picking substring till indexed character
            seq2=seq_list[i+2] # picking line4 example: .((((.......)))).........................
            seq2_indx=seq2.index(char) #finding position of first index character
            seq2_substr = seq2[:seq2_indx] #picking till indexed character
            if seq1_substr==seq2_substr: #check if both lines are same or not
                list1.append(seq_list[i-2]) #appending line2
                list2.append(seq_list[i]) #appending line1
    df = pd.DataFrame(list(zip(list1, list2)),columns =['Brk', 'val']) #Calling DataFrame after zipping both lists with names
    #df = pd.DataFrame(list(zip(list1, list2)),columns =['Brk', 'val']) #Calling DataFrame after zipping both lists with names
    df[['sep','Sliding_window']] = df["Brk"].str.split(">",n=1, expand = True)
    #Split the 'Brk' column string by spaces ' ', then take the second part (index 1),and split this part by '_' and take the first part (index 0)
    df['StartB'] = df['Brk'].str.split(' ').str[1].str.split('_').str[0]
    # Split the 'val' column string by '(' first, then by ')', and finally by ' ', we use rsplit to split from the right and only split once (maxsplit parameter is set to 1)
    df['MFE'] = df['val'].str.rsplit('(', n=1).str[1].str.split(')').str[0].str.strip()
    df['MFE'] = df['MFE'].astype(str).str.replace('-', '')
    df['MFE'] = df['MFE'].astype(str).astype(float)
    df['ranked']=df.groupby('StartB')['MFE'].rank('dense',ascending=False) # grouping based on breakpoint and choosing the one with lowest MFE
    df_final=df[df['ranked']==1] #(We only pick those whose MFE is lowest)
    # Select the columns of interest
    Column_of_Interest = df_final.loc[:, ['StartB', 'MFE', 'ranked', 'Sliding_window']]
    Drp_duplicates_column_of_Intrst_minus = Column_of_Interest.drop_duplicates(subset='StartB', keep='first') #Only keep top values of MFE and drop other values
    Drp_duplicates_column_of_Intrst_minus = pd.DataFrame(Drp_duplicates_column_of_Intrst_minus)
    outfile_path = os.path.join(output_folder, "Drp_duplicates_column_of_Intrst_minus.csv")
    # Save the data_plus DataFrame as a CSV file
    Drp_duplicates_column_of_Intrst_minus.to_csv(outfile_path, index=False)
    
    from rpy2 import robjects
    from rpy2.robjects import pandas2ri

    # Set the R code
    r_code = '''
    library("ggplot2") 
    library("vctrs")
    library("dplyr")

    # Read the data files from the output folder
    data1 <- read.csv(file.path(output_folder, "PROPORTION_enda.csv"))
    data2 <- read.csv(file.path(output_folder, "Drp_duplicates_column_of_Intrst_plus.csv"))
    data3 <- read.csv(file.path(output_folder, "PROPORTION_startb.csv"))
    data4 <- read.csv(file.path(output_folder, "Drp_duplicates_column_of_Intrst_minus.csv"))

    Merged_data_frame <- left_join(data1, data2, by = 'EndA')
    Merged_data_frame <- select(Merged_data_frame, c("EndA", "Count", "Proportion", "MFE"))
    Plus_dataframe <- replace(Merged_data_frame, is.na(Merged_data_frame), 0)
    Plus_dataframe <- mutate(Plus_dataframe, Category = "Coding / Plus strand")
    Plus_dataframe <- select(Plus_dataframe, c("EndA", "Count", "Proportion", "MFE", "Category"))
    colnames(Plus_dataframe) <- c("Breakpoint", "Count", "Proportion", "MFE", "Category")

    Merged_data_frame <- left_join(data3, data4, by = 'StartB')
    Merged_data_frame <- select(Merged_data_frame, c("StartB", "Count", "Proportion", "MFE"))
    Minus_dataframe <- replace(Merged_data_frame, is.na(Merged_data_frame), 0)
    Minus_dataframe <- mutate(Minus_dataframe, Category = "Non-Coding / Minus strand")
    Minus_dataframe <- select(Minus_dataframe, c("StartB", "Count", "Proportion", "MFE", "Category"))
    colnames(Minus_dataframe) <- c("Breakpoint", "Count", "Proportion", "MFE", "Category")

    # Specify the path to the FASTA file
    fasta_file <- file.path(output_folder, "payload_plusstrand.fasta")
    # Read the FASTA file
    fasta_content <- readLines(fasta_file)
    # Remove header lines starting with ">"
    fasta_content <- fasta_content[!startsWith(fasta_content, ">")]
    # Calculate the total base count
    base_count <- sum(nchar(fasta_content)) + 100

    # Bind both dataframes
    plus_minus_df <- rbind(Plus_dataframe, Minus_dataframe)
    write.csv(plus_minus_df, file.path(output_folder, "plus_minus_MFE.csv"))
    max_proportion <- max(plus_minus_df$Proportion)
    max_proportion <- max_proportion + 0.05 #this is just been added to make y axis look wider so that the figure looks good.
    max_count <- max(plus_minus_df$Count)
    max_count <- max_count + 100 #this is just been added to make y axis look wider so that the figure looks good.

    # Plot two graphs together with spacing
    start_coo <- 0
    tot_len <- base_count + 1000
    Sheet_plot <- ggplot(plus_minus_df, aes(x = Breakpoint , y = Proportion, fill = MFE)) +
        geom_bar(stat = "identity", width = 20)+  theme(axis.text.x = element_text(angle = 90, size = 10)) + 
        theme(panel.grid.major = element_blank(),panel.grid.minor = element_blank()) + 
        scale_x_continuous(limits = c(start_coo,tot_len), expand = c(0, 0)) + 
        scale_y_continuous(limits = c(start_coo,max_proportion),labels = scales::percent_format(accuracy = 1),expand = c(0, 0)) + 
        theme_bw() + labs(main = " DNA_strand",y = "Proportion", x= "Snapback coordinates") +
        scale_fill_gradient(low = "navyblue", high = "yellow")+ 
        facet_wrap(~ Category) +
        theme(panel.spacing = unit(0.5, "cm", data = NULL))
    Sheet_plot1 <- ggplot(plus_minus_df, aes(x = Breakpoint , y = Count, fill = MFE)) +
        geom_bar(stat = "identity", width = 20)+  theme(axis.text.x = element_text(angle = 90, size = 10)) + 
        theme(panel.grid.major = element_blank(),panel.grid.minor = element_blank()) + 
        scale_x_continuous(limits = c(start_coo,tot_len), expand = c(0, 0)) + 
        scale_y_continuous(limits = c(start_coo,max_count),expand = c(0, 0)) + 
        theme_bw() + labs(main = " DNA_strand",y = "Count", x= "Snapback coordinates") +
        scale_fill_gradient(low = "navyblue", high = "yellow")+ 
        facet_wrap(~ Category) +
        theme(panel.spacing = unit(0.5, "cm", data = NULL))


    pdf(file.path(output_folder, "ggplot.pdf"))
    print(Sheet_plot)
    print(Sheet_plot1)
    dev.off()
    '''

    # Set the R output folder variable
    robjects.globalenv['output_folder'] = str(output_folder)

    # Execute the R code
    robjects.r(r_code)

## 4. Secondary structure consistency validation

This validation step compares RNAfold-predicted secondary structures and centroid structures to retain only breakpoint windows with consistent structural predictions for downstream MFE analysis.

In [ ]:
# Unit Test: Checking for similar secondary structures
plus_rnafold_filepath = os.path.join(output_folder, "Comb_plus_rnafold.seq")
with open(plus_rnafold_filepath, 'r') as fd:
    seq_list = fd.readlines() 
    list1 = [] #creating empty list 1
    list2 = [] #creating empty list 2
    char = " " #character based on which we seperate the lines
    for i in range(2,len(seq_list),6): #creating loop and picking every 2,3,4 line from each batch
        seq1=seq_list[i] # picking line2 example: .((((.......))))......................... 
        seq1_indx=seq1.index(char) #finding position of first index character
        seq1_substr = seq1[:seq1_indx] #picking substring till indexed character
        seq2=seq_list[i+2] # picking line4 example: .((((.......)))).........................
        seq2_indx=seq2.index(char) #finding position of first index character
        seq2_substr = seq2[:seq2_indx] #picking till indexed character
        if seq1_substr==seq2_substr: #check if both lines are same or not
            list1.append(seq1_substr) #appending line2
            list2.append(seq2_substr) #appending line1
# Now you can perform assertions or other checks on list1 and list2 to validate the unit test.
# For example, you can check if both lists have the same length, indicating that they contain similar secondary structures.
# You can also compare specific elements of the lists to verify their similarity.
# Feel free to add assertions or other checks based on your testing requirements.
## Unit test
import unittest

class TestStringMethods(unittest.TestCase):

    def test_upper(self):
        self.assertEqual(list1, list2)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False) 

## 5. Add breakpoint nucleotide identity

This section adds the nucleotide present at each breakpoint position from the plus and minus strand payload sequences.

In [ ]:
from pathlib import Path
import pandas as pd
from Bio import SeqIO

workdir = OUTPUT_FOLDER
# --- paths ---
tbl_in = workdir / "plus_minus_MFE.csv"
fasta_plus = workdir / "payload_plusstrand.fasta"
fasta_minus = workdir / "payload_minusstrand.fasta"

# --- helpers ---
def read_payload_seq(fa_path):
    seq = ""
    for rec in SeqIO.parse(fa_path, "fasta"):
        seq += str(rec.seq)
    return seq.upper()

def get_base_1based(seq, pos):
    if pd.isna(pos):
        return "N"
    p = int(pos)
    return seq[p-1] if 1 <= p <= len(seq) else "N"

# --- load inputs ---
df = pd.read_csv(tbl_in)
df["Breakpoint"] = df["Breakpoint"].astype(int)
df["Strand"] = df["Category"].apply(lambda s: "plus" if str(s).startswith("Coding") else "minus")

# Load payload sequences
plus_seq  = read_payload_seq(fasta_plus)
minus_seq = read_payload_seq(fasta_minus)

# Add Base column
df["Base"] = df.apply(
    lambda r: get_base_1based(plus_seq if r["Strand"]=="plus" else minus_seq, r["Breakpoint"]),
    axis=1
)

# --- write annotated table with base column ---
out_annot = workdir / "plus_minus_MFE_with_bases.csv"
df.to_csv(out_annot, index=False)
print(f"[✓] Wrote: {out_annot}")

# --- nucleotide counts per strand ---
strand_counts = []
for strand, subdf in df.groupby("Strand"):
    counts = (
        subdf.groupby("Base")["Count"]
             .sum()
             .rename("Count")
             .reset_index()
    )
    valid = {"A","C","G","T"}
    counts["Base"] = counts["Base"].str.upper().replace({b: b for b in valid})
    counts.loc[~counts["Base"].isin(list(valid)), "Base"] = "N"
    counts = counts.groupby("Base", as_index=False)["Count"].sum()
    total = counts["Count"].sum()
    counts["Proportion"] = counts["Count"] / total if total else 0.0
    counts.insert(0, "Strand", strand)
    strand_counts.append(counts)

final_counts = pd.concat(strand_counts, ignore_index=True)
order = ["A","C","G","T","N"]
final_counts["Base"] = pd.Categorical(final_counts["Base"], categories=order, ordered=True)
final_counts = final_counts.sort_values(["Strand","Base"]).reset_index(drop=True)

# --- write counts file ---
out_counts = workdir / "nucleotide_counts_by_strand.csv"
final_counts.to_csv(out_counts, index=False)
print(f"[✓] Wrote: {out_counts}")

# quick look
display(final_counts)


## 6. Nucleotide enrichment analysis

This section compares observed breakpoint nucleotide counts with the payload background nucleotide composition using a chi-square test.

In [ ]:
import os
import pandas as pd
from Bio import SeqIO
from scipy.stats import chisquare
import numpy as np

# ---- input files (under workdir) ----
count_file = os.path.join(workdir, "nucleotide_counts_by_strand.csv")
fasta_plus  = os.path.join(workdir, "payload_plusstrand.fasta")
fasta_minus = os.path.join(workdir, "payload_minusstrand.fasta")

# ---- helpers ----
def read_seq(fasta_path):
    seq = []
    for rec in SeqIO.parse(fasta_path, "fasta"):
        seq.append(str(rec.seq).upper())
    return "".join(seq)

def background_freq(seq):
    total = len(seq)
    # guard: avoid division by zero
    if total == 0:
        return {"A":0, "C":0, "G":0, "T":0}
    return {
        "A": seq.count("A")/total,
        "C": seq.count("C")/total,
        "G": seq.count("G")/total,
        "T": seq.count("T")/total,
    }

# ---- load data ----
df = pd.read_csv(count_file)  # expects columns: Base, Count, Strand
# normalize base labels
df["Base"] = df["Base"].str.upper()
df = df[df["Base"].isin(list("ACGT"))]

# ---- background from payloads ----
plus_seq  = read_seq(fasta_plus)
minus_seq = read_seq(fasta_minus)

bg_plus  = background_freq(plus_seq)
bg_minus = background_freq(minus_seq)

bases = ["A","C","G","T"]

# ---- run chi-square per strand ----
rows_summary  = []
rows_detailed = []

for strand, sub in df.groupby("Strand"):
    # observed
    observed = np.array([sub.loc[sub["Base"]==b, "Count"].sum() for b in bases], dtype=float)
    total_obs = observed.sum()

    # expected from background
    bg = bg_plus if strand.lower() == "plus" else bg_minus
    expected = np.array([bg[b]*total_obs for b in bases], dtype=float)

    # chi-square GOF
    chi2, p = chisquare(f_obs=observed, f_exp=expected)

    # standardized residuals
    with np.errstate(divide='ignore', invalid='ignore'):
        std_resid = (observed - expected) / np.sqrt(expected)
        std_resid = np.where(expected > 0, std_resid, np.nan)

    # save per-strand summary row
    rows_summary.append({
        "Strand": strand,
        "TotalBreakpoints": int(total_obs),
        "Chi2": round(float(chi2), 3),
        "p_value": float(p),
        "bg_A": round(bg["A"], 4),
        "bg_C": round(bg["C"], 4),
        "bg_G": round(bg["G"], 4),
        "bg_T": round(bg["T"], 4),
    })

    # save detailed rows
    for i, b in enumerate(bases):
        rows_detailed.append({
            "Strand": strand,
            "Base": b,
            "Observed": int(observed[i]),
            "Expected": round(float(expected[i]), 3),
            "StdResidual": round(float(std_resid[i]), 3)
        })

summary_df  = pd.DataFrame(rows_summary)
detailed_df = pd.DataFrame(rows_detailed)

# ---- write outputs ----
summary_path  = os.path.join(workdir, "chi_square_results_by_strand.csv")
detailed_path = os.path.join(workdir, "chi_square_detailed_by_strand.csv")
summary_df.to_csv(summary_path, index=False)
detailed_df.to_csv(detailed_path, index=False)

print("Wrote:", summary_path)
print("Wrote:", detailed_path)
print(summary_df)

## 7. Generate final publication-ready plots

This section merges plus and minus strand breakpoint data with MFE values and generates final visualization outputs in PNG, PDF, and TIFF formats for publication use.

In [ ]:
import os
from rpy2 import robjects
from rpy2.robjects import StrVector

# =========================================================
# SET YOUR SAMPLE-SPECIFIC OUTPUT FOLDER HERE
# =========================================================
output_folder = str(OUTPUT_FOLDER)

# =========================================================
# CHECK INPUT FILES
# =========================================================
print("Output folder:", output_folder)
print("Exists:", os.path.exists(output_folder))

required_files = [
    "PROPORTION_enda.csv",
    "Drp_duplicates_column_of_Intrst_plus.csv",
    "PROPORTION_startb.csv",
    "Drp_duplicates_column_of_Intrst_minus.csv"
]

for f in required_files:
    print(f, os.path.exists(os.path.join(output_folder, f)))

# =========================================================
# PASSING FOLDER TO R
# =========================================================
robjects.globalenv["output_folder"] = StrVector([output_folder])

# =========================================================
# R CODE
# =========================================================
r_code = '''
library(ggplot2)
library(dplyr)
library(scales)
library(viridis)

# Make sure output_folder is plain text
output_folder <- as.character(output_folder)[1]

cat("Output folder in R:", output_folder, "\\n")
cat("Current working directory:", getwd(), "\\n")

# Show files already present
cat("Files before writing:\\n")
print(list.files(output_folder))

# Read input files
data1 <- read.csv(paste0(output_folder, "/PROPORTION_enda.csv"))
data2 <- read.csv(paste0(output_folder, "/Drp_duplicates_column_of_Intrst_plus.csv"))
data3 <- read.csv(paste0(output_folder, "/PROPORTION_startb.csv"))
data4 <- read.csv(paste0(output_folder, "/Drp_duplicates_column_of_Intrst_minus.csv"))

# Plus strand
merged_plus <- left_join(data1, data2, by = "EndA")
merged_plus <- select(merged_plus, EndA, Count, Proportion, MFE)
plus_df <- replace(merged_plus, is.na(merged_plus), 0)
plus_df <- mutate(plus_df, Category = "Coding / Plus strand")
colnames(plus_df) <- c("Breakpoint", "Count", "Proportion", "MFE", "Category")

# Minus strand
merged_minus <- left_join(data3, data4, by = "StartB")
merged_minus <- select(merged_minus, StartB, Count, Proportion, MFE)
minus_df <- replace(merged_minus, is.na(merged_minus), 0)
minus_df <- mutate(minus_df, Category = "Non-Coding / Minus strand")
colnames(minus_df) <- c("Breakpoint", "Count", "Proportion", "MFE", "Category")

# Combine
plus_minus_df <- rbind(plus_df, minus_df)

# Save combined CSV in same folder
write.csv(
    plus_minus_df,
    paste0(output_folder, "/plus_minus_MFE.csv"),
    row.names = FALSE
)

# Axis limits
max_proportion <- max(plus_minus_df$Proportion, na.rm = TRUE) + 0.05
max_count <- max(plus_minus_df$Count, na.rm = TRUE) * 1.1

# Plot 1: Proportion
p1 <- ggplot(plus_minus_df, aes(x = Breakpoint, y = Proportion, fill = MFE)) +
    geom_bar(stat = "identity", width = 20) +
    scale_y_continuous(
        limits = c(0, max_proportion),
        labels = percent_format(accuracy = 1),
        expand = c(0, 0)
    ) +
    scale_x_continuous(expand = c(0, 0)) +
    scale_fill_viridis(option = "C", direction = 1) +
    facet_wrap(~ Category) +
    labs(
        title = "Breakpoint Distribution (Proportion)",
        x = "Snapback Coordinate",
        y = "Proportion",
        fill = "MFE"
    ) +
    theme_classic(base_size = 14) +
    theme(
        axis.text.x = element_text(angle = 90, size = 8),
        strip.text = element_text(size = 12, face = "bold"),
        plot.title = element_text(hjust = 0.5)
    )

# Plot 2: Count
p2 <- ggplot(plus_minus_df, aes(x = Breakpoint, y = Count, fill = MFE)) +
    geom_bar(stat = "identity", width = 20) +
    scale_y_continuous(
        limits = c(0, max_count),
        expand = c(0, 0)
    ) +
    scale_x_continuous(expand = c(0, 0)) +
    scale_fill_viridis(option = "C", direction = 1) +
    facet_wrap(~ Category) +
    labs(
        title = "Breakpoint Distribution (Count)",
        x = "Snapback Coordinate",
        y = "Count",
        fill = "MFE"
    ) +
    theme_classic(base_size = 14) +
    theme(
        axis.text.x = element_text(angle = 90, size = 8),
        strip.text = element_text(size = 12, face = "bold"),
        plot.title = element_text(hjust = 0.5)
    )

# Save all outputs in same folder
ggsave(
    filename = paste0(output_folder, "/Figure_Proportion.png"),
    plot = p1,
    width = 14,
    height = 6,
    dpi = 300
)

ggsave(
    filename = paste0(output_folder, "/Figure_Count.png"),
    plot = p2,
    width = 14,
    height = 6,
    dpi = 300
)

ggsave(
    filename = paste0(output_folder, "/Figure_Proportion.pdf"),
    plot = p1,
    width = 14,
    height = 6
)

ggsave(
    filename = paste0(output_folder, "/Figure_Count.pdf"),
    plot = p2,
    width = 14,
    height = 6
)

# =========================
# TIFF OUTPUT (ADD HERE)
# =========================

ggsave(
    filename = paste0(output_folder, "/Figure_Proportion.tiff"),
    plot = p1,
    width = 14,
    height = 6,
    dpi = 600,
    device = "tiff",
    compression = "lzw"
)

ggsave(
    filename = paste0(output_folder, "/Figure_Count.tiff"),
    plot = p2,
    width = 14,
    height = 6,
    dpi = 600,
    device = "tiff",
    compression = "lzw"
)

pdf(paste0(output_folder, "/ggplot_updated.pdf"), width = 14, height = 12)
print(p1)
print(p2)
dev.off()

cat("Files after writing:\\n")
print(list.files(output_folder))
'''

# =========================================================
# RUN R CODE
# =========================================================
robjects.r(r_code)

# =========================================================
# CHECK OUTPUT FILES IN PYTHON
# =========================================================
print("\\nFiles after R run:")
for f in [
    "plus_minus_MFE.csv",
    "Figure_Proportion.png",
    "Figure_Count.png",
    "Figure_Proportion.pdf",
    "Figure_Count.pdf",
    "ggplot_updated.pdf"
]:
    print(f, os.path.exists(os.path.join(output_folder, f)))